# 🔐 CodeBERT — Vuln Detector v4 (function-level)

Train trên `detect_v4_functionlevel.jsonl` (3,558 hàm, cân bằng, đã khử confound + chống rò rỉ).
Function-level (~93% ≤512 token) → **không cần chunking/MIL**. CodeBERT + head phân loại nhị phân, tinh gọn.

**Kaggle:** Add Data → upload `detect_v4_functionlevel.jsonl` · Bật GPU · Run cài đặt → **Restart kernel** → Run All.

In [1]:
# Cài bản ổn định cho Kaggle, rồi RESTART KERNEL
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "accelerate==1.0.1" scikit-learn
print("Xong -> Run -> Restart Kernel -> Run All")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 549.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 24.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.25.2 which is incompatible.
Xong -> Run -> Restart Kernel -> Run All


## 1. Cấu hình

In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json, glob, random, collections
import numpy as np, torch
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "microsoft/codebert-base"
MAX_LEN    = 512
TRAIN_BS   = 16      # OOM? -> 8
EVAL_BS    = 32
EPOCHS     = 8
LR         = 2e-5
WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 0.01, 0.06, 3
LABEL2ID = {"Safe": 0, "Vulnerable": 1}; ID2LABEL = {0: "Safe", 1: "Vulnerable"}
OUTPUT_DIR, SAVE_DIR = "./codebert-v4", "./codebert-v4-final"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("transformers", transformers.__version__, "| device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

transformers 4.46.3 | device: Tesla T4


## 2. Nạp dữ liệu (dùng sẵn split train/val/test trong file)

In [3]:
def find_data():
    for p in ["/kaggle/input/**/detect_v4_functionlevel.jsonl",
              "detect_v4_functionlevel.jsonl",
              "../DatasetBuild/output/detect_v4_functionlevel.jsonl",
              "DatasetBuild/output/detect_v4_functionlevel.jsonl"]:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    raise FileNotFoundError("Chưa thấy data -> Add Data: upload detect_v4_functionlevel.jsonl")

PATH = find_data(); print("DATA:", PATH)
rows = [json.loads(l) for l in open(PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABEL2ID]

splits = {"train": [], "val": [], "test": []}
for r in rows: splits.get(r.get("split", "train"), splits["train"]).append(r)
for k, v in splits.items():
    nv = sum(x["label"] == "Vulnerable" for x in v)
    print(f"  {k:5s}: {len(v):5d} (Vuln {nv}/Safe {len(v)-nv}) | {dict(collections.Counter(x['source'] for x in v))}")

DATA: /kaggle/input/datasets/thanhphuocjr/detect-v4-functionlevel/detect_v4_functionlevel.jsonl
  train:  2848 (Vuln 1429/Safe 1419) | {'solodit': 1732, 'dappscan': 1116}
  val  :   355 (Vuln 178/Safe 177) | {'solodit': 203, 'dappscan': 152}
  test :   355 (Vuln 172/Safe 183) | {'solodit': 215, 'dappscan': 140}


## 3. Tokenize (truncation 512)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.enc = tokenizer([r["code"] for r in items], truncation=True, max_length=MAX_LEN)
        self.y   = [LABEL2ID[r["label"]] for r in items]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: self.enc[k][i] for k in self.enc}; d["labels"] = self.y[i]; return d

train_ds, val_ds, test_ds = DS(splits["train"]), DS(splits["val"]), DS(splits["test"])
collator = DataCollatorWithPadding(tokenizer)
ntr = sum(1 for r in splits["train"] if len(tokenizer(r["code"])["input_ids"]) > MAX_LEN)
print(f"Train bị cắt >512 token: {ntr}/{len(splits['train'])} ({100*ntr/len(splits['train']):.1f}%)")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (569 > 512). Running this sequence through the model will result in indexing errors


Train bị cắt >512 token: 387/2848 (13.6%)


## 4. Mô hình + Huấn luyện

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID).to(device)

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=-1); y = p.label_ids
    P, R, F, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pr), "precision": P, "recall": R,
            "f1_macro": F, "f1_vuln": f1_score(y, pr, pos_label=1, zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine", fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collator, tokenizer=tokenizer, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
trainer.train()

print("\nEpoch | val_acc | val_prec | val_recall | val_f1 | val_f1_vuln")
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        print(f"  {round(h['epoch']):3d} | {h['eval_accuracy']:.4f} | {h['eval_precision']:.4f} | "
              f"{h['eval_recall']:.4f} | {h['eval_f1_macro']:.4f} | {h['eval_f1_vuln']:.4f}")

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_58/3401052291.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro,F1 Vuln
1,0.705500,0.675615,0.591549,0.709140,0.592601,0.526017,0.349776
2,0.635300,0.577512,0.681690,0.682695,0.681791,0.681326,0.670554
3,0.513200,0.560131,0.738028,0.738948,0.738113,0.737820,0.730435
4,0.362400,0.600283,0.726761,0.727117,0.726814,0.726682,0.722063
5,0.294500,0.657755,0.752113,0.752913,0.752190,0.751953,0.745665
6,0.198000,0.765666,0.732394,0.733294,0.732480,0.732182,0.724638
7,0.150500,0.787390,0.752113,0.753267,0.752206,0.751874,0.744186
8,0.131100,0.788773,0.749296,0.749936,0.749365,0.749168,0.743516


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


Epoch | val_acc | val_prec | val_recall | val_f1 | val_f1_vuln
    1 | 0.5915 | 0.7091 | 0.5926 | 0.5260 | 0.3498
    2 | 0.6817 | 0.6827 | 0.6818 | 0.6813 | 0.6706
    3 | 0.7380 | 0.7389 | 0.7381 | 0.7378 | 0.7304
    4 | 0.7268 | 0.7271 | 0.7268 | 0.7267 | 0.7221
    5 | 0.7521 | 0.7529 | 0.7522 | 0.7520 | 0.7457
    6 | 0.7324 | 0.7333 | 0.7325 | 0.7322 | 0.7246
    7 | 0.7521 | 0.7533 | 0.7522 | 0.7519 | 0.7442
    8 | 0.7493 | 0.7499 | 0.7494 | 0.7492 | 0.7435


## 5. Đánh giá TEST — Accuracy / Precision / Recall / F1

In [6]:
pred = trainer.predict(test_ds)
yp = np.argmax(pred.predictions, axis=-1); yt = pred.label_ids

print("=" * 58 + f"\nTEST (n={len(yt)})\n" + "=" * 58)
print(classification_report(yt, yp, target_names=["Safe", "Vulnerable"], digits=4, zero_division=0))
print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(yt, yp))
P, R, F, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
print(f"\n>> Accuracy={accuracy_score(yt, yp):.4f}  Precision={P:.4f}  Recall={R:.4f}  F1={F:.4f}")

# theo nguồn (trung thực: nguồn nào model yếu)
src = np.array([r["source"] for r in splits["test"]])
print("\n--- Theo nguồn ---")
print(f"{'source':10s} {'n':>4s} {'acc':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s}")
for s in sorted(set(src)):
    ix = np.where(src == s)[0]
    p, r, f, _ = precision_recall_fscore_support(yt[ix], yp[ix], average="macro", zero_division=0)
    print(f"{s:10s} {len(ix):4d} {accuracy_score(yt[ix], yp[ix]):7.3f} {p:7.3f} {r:7.3f} {f:7.3f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TEST (n=355)
              precision    recall  f1-score   support

        Safe     0.7347    0.7869    0.7599       183
  Vulnerable     0.7547    0.6977    0.7251       172

    accuracy                         0.7437       355
   macro avg     0.7447    0.7423    0.7425       355
weighted avg     0.7444    0.7437    0.7430       355

Confusion [[TN FP][FN TP]]:
 [[144  39]
 [ 52 120]]

>> Accuracy=0.7437  Precision=0.7447  Recall=0.7423  F1=0.7425

--- Theo nguồn ---
source        n     acc    prec  recall      f1
dappscan    140   0.621   0.621   0.622   0.621
solodit     215   0.823   0.826   0.818   0.820


## 6. Lưu mô hình

In [7]:
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
print("Đã lưu:", SAVE_DIR, os.listdir(SAVE_DIR))

Đã lưu: ./codebert-v4-final ['special_tokens_map.json', 'training_args.bin', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'merges.txt', 'vocab.json', 'tokenizer.json']


## 7. Thử inference

In [8]:
model.eval()
@torch.no_grad()
def predict(code):
    enc = tokenizer(code, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    p = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if p >= 0.5 else "Safe"), p

TEST_CODE = "function withdraw(uint amount) public {\n    require(balances[msg.sender] >= amount);\n    (bool ok,) = msg.sender.call{value: amount}(\"\");\n    require(ok);\n    balances[msg.sender] -= amount;\n}"
lbl, prob = predict(TEST_CODE)
print(f"Dự đoán: {lbl}  (P(Vulnerable)={prob:.3f})")

Dự đoán: Vulnerable  (P(Vulnerable)=0.731)
